# Jev — render the explainer in 2K on Google Colab

This notebook renders the **13:14 Jev explainer** at **2560×1440 (2K)** with Remotion, then saves the MP4 and render log to **Google Drive**.

**Read this first**

1. **Google Drive access is requested first.** The next cell calls `drive.mount()` and will not continue until you approve the Google permission prompt. Nothing is cloned, installed, or rendered before that.
2. The output folder defaults to `/content/drive/MyDrive/jev_render`. You can change `DRIVE_DIR` in the Drive cell.
3. 2K rendering is **CPU-bound**. On a free Colab runtime this master can take several hours; a runtime with more CPUs (Colab Pro) will be faster. A GPU does **not** speed up Remotion.
4. Keep this tab open while rendering. The cell prints live progress and ETA.
5. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) with read access before running the clone cell.

## 1) Google Drive access (required)

Approve the Google Drive permission prompt. The notebook refuses to render until Drive is mounted.

In [ ]:
#@title 1) Connect Google Drive first
from google.colab import drive
import os

DRIVE_DIR = "/content/drive/MyDrive/jev_render"  #@param {type:"string"}

print("Requesting Google Drive access. Approve the prompt to continue...")
drive.mount("/content/drive", force_remount=False)

if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive was not mounted. Re-run this cell and approve access.")

os.makedirs(DRIVE_DIR, exist_ok=True)
probe = os.path.join(DRIVE_DIR, ".write-test")
with open(probe, "w") as f:
    f.write("ok")
os.remove(probe)
print("Drive ready:", DRIVE_DIR)

## 2) Configuration

The default is a true 2K render: the 1920×1080 master is rendered at **1.3333× scale**, giving **2560×1440**.

`TEST_SECONDS = 8` renders an 8-second check first. Set it back to `0` for the full video.

In [ ]:
#@title 2) Render settings
import os

REPO_HTTPS = "https://github.com/lroe/jev.git"   #@param {type:"string"}
BRANCH = "main"                                  #@param {type:"string"}
REPO_DIR = "/content/jev"                        #@param {type:"string"}

SCALE = 2560 / 1920                              # true 2K = 2560x1440
CONCURRENCY = max(2, min(4, os.cpu_count() or 2))
CRF = 16                                         # lower = better quality / bigger file
TEST_SECONDS = 0                                 # 8 for a quick test, 0 for full
OUT_NAME = "jev_2k.mp4" if TEST_SECONDS == 0 else "jev_2k_test.mp4"

print(f"Resolution: {round(1920*SCALE)}x{round(1080*SCALE)}")
print(f"Concurrency: {CONCURRENCY} (CPU count: {os.cpu_count()})")
print(f"Mode: {'full 13:14' if TEST_SECONDS == 0 else f'test {TEST_SECONDS}s'}")
print(f"Output: {os.path.join(DRIVE_DIR, OUT_NAME)}")

## 3) Install Node and Chromium libraries

Colab changes its base image occasionally. This installs Node 22 if the runtime has an older Node, then installs the system libraries used by headless Chromium.

In [ ]:
#@title 3) Node + system libraries
import shutil, subprocess

def node_major():
    try:
        return int(subprocess.check_output(["node", "-v"]).decode().strip().lstrip("v").split(".")[0])
    except Exception:
        return 0

print("Node before:", shutil.which("node") or "not found")
if node_major() < 20:
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -", shell=True, check=True)
    subprocess.run("sudo apt-get install -y nodejs", shell=True, check=True)
else:
    print("Node is new enough, skipping install.")

print(subprocess.check_output(["node", "-v"]).decode().strip())
print(subprocess.check_output(["npm", "-v"]).decode().strip())

# Headless Chromium dependencies. These are present on most Colab images but
# installing them explicitly avoids "error while loading shared libraries".
subprocess.run("sudo apt-get update -qq", shell=True, check=False)
subprocess.run(
    "sudo apt-get install -y -qq libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 "
    "libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 "
    "libgbm1 libasound2 libpango-1.0-0 libcairo2",
    shell=True,
    check=False,
)
print("System libraries ready.")

## 4) Clone the project

The public HTTPS URL is tried first. If the repo is private, the cell uses a `GITHUB_TOKEN` Colab secret. No token is stored in this notebook.

In [ ]:
#@title 4) Clone / update GitHub repo
import os, shutil, subprocess
from google.colab import userdata

token = None
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

urls = []
if token:
    urls.append(f"https://{token}@github.com/lroe/jev.git")
urls.append(REPO_HTTPS)

if os.path.isdir(REPO_DIR):
    print("Removing old checkout:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

ok = False
for i, url in enumerate(urls):
    label = "GitHub (token)" if i == 0 and token else REPO_HTTPS
    print("Cloning", label, "→", REPO_DIR)
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], check=True)
        ok = True
        break
    except subprocess.CalledProcessError as e:
        print("Clone failed, exit code", e.returncode)

if not ok:
    raise RuntimeError(
        "Could not clone the repo. If it is private, add a Colab secret named "
        "GITHUB_TOKEN with repo read access (key icon on the left), then re-run this cell."
    )

os.chdir(REPO_DIR)
print("Checked out:", subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())

## 5) Install npm dependencies and Chrome

This installs the pinned Remotion stack and downloads the headless browser used for rendering. It is safe to re-run.

In [ ]:
#@title 5) Install dependencies
import os, subprocess

PROJECT = os.path.join(REPO_DIR, "video")
os.chdir(PROJECT)
print("Project:", PROJECT)

subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], check=True)
subprocess.run(["npx", "remotion", "browser", "ensure"], check=True)
print("Installed. Node modules and Chrome are ready.")

## 6) Render 2K with live progress

This is the long cell.

- Remotion writes to `/content/jev_2k.mp4` first.
- Progress is streamed line-by-line from the renderer (`onProgress`) and printed below.
- When the render finishes, the MP4 and `render.log` are copied to `DRIVE_DIR`.
- Do not close the tab. If Colab disconnects, the local MP4 is lost; the render would need to restart.

The default `SCALE` produces **2560×1440**. Change `SCALE = 1` in the config cell for a 1080p test.

In [ ]:
#@title 6) Render + save to Google Drive
import os, re, shutil, subprocess, time

if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive is not mounted. Run cell 1 first and approve access.")

PROJECT = os.path.join(REPO_DIR, "video")
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
os.makedirs(DRIVE_DIR, exist_ok=True)

test = TEST_SECONDS > 0
local_out = "/content/jev_2k_test.mp4" if test else "/content/jev_2k.mp4"
log_local = "/content/jev_2k_render.log"

cmd = [
    "node", "scripts/render_2k.mjs",
    "--output", local_out,
    "--scale", str(SCALE),
    "--concurrency", str(CONCURRENCY),
    "--crf", str(CRF),
]
if test:
    cmd += ["--frames", f"0-{int(TEST_SECONDS * 30) - 1}"]

print("Running:", " ".join(cmd))
print("Live progress starts below. This can take a long time on CPU.\n", flush=True)

start = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=PROJECT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

with open(log_local, "w") as log:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
code = proc.wait()
elapsed = time.time() - start

log_drive = os.path.join(DRIVE_DIR, os.path.basename(log_local))
shutil.copy2(log_local, log_drive)
print(f"\nRender log saved to: {log_drive}")

if code != 0:
    raise RuntimeError(f"Render failed with exit code {code}. See {log_drive}")

if not os.path.isfile(local_out):
    raise RuntimeError("Render finished but no MP4 was found.")

drive_out = os.path.join(DRIVE_DIR, os.path.basename(local_out))
print(f"Copying MP4 to Drive: {drive_out}")
shutil.copy2(local_out, drive_out)
size_mb = os.path.getsize(drive_out) / (1024 * 1024)

print("\n" + "=" * 64)
print(f"Done in {elapsed / 60:.1f} min")
print(f"2K MP4: {drive_out}  ({size_mb:.1f} MB)")
print(f"Log:    {log_drive}")
print("=" * 64)

## 7) Verify the output

Run this after the render. It prints the file path, size, and the actual video dimensions/duration from `ffprobe`.

In [ ]:
#@title 7) Inspect the rendered file
import os, subprocess

drive_out = os.path.join(DRIVE_DIR, "jev_2k_test.mp4" if TEST_SECONDS > 0 else "jev_2k.mp4")
print("File:", drive_out)
print("Exists:", os.path.isfile(drive_out))
if os.path.isfile(drive_out):
    print("Size MB:", round(os.path.getsize(drive_out) / (1024 * 1024), 1))
    print(subprocess.check_output([
        "ffprobe", "-v", "error",
        "-show_entries", "stream=codec_name,width,height,avg_frame_rate",
        "-show_entries", "format=duration,size",
        "-of", "default=noprint_wrappers=1",
        drive_out,
    ]).decode())
else:
    print("Render the video first (cell 6).")

# Optional inline preview. A full 13-minute 2K file is large; open it in Drive
# instead if this is slow.
# from IPython.display import Video
# display(Video(drive_out, embed=False, width=960))

## Troubleshooting

- **Drive prompt not approved** → re-run cell 1 and approve access. Nothing renders without it.
- **Private repo clone fails** → add a Colab secret named `GITHUB_TOKEN` with read access (key icon on the left), then re-run cell 4.
- **`Could not find Chrome` / missing shared library** → re-run cells 3 and 5 (`npx remotion browser ensure`).
- **Out of memory / disk** → use a high-RAM runtime, remove old files under `/content`, or render a shorter `TEST_SECONDS` clip first.
- **Colab disconnects** → the local MP4 is lost. The render must be restarted. Colab Pro or a longer-lived runtime is recommended for the full 2K master.
- **Rendering is taking hours** → that is expected for 2K. More CPUs help; GPUs do not. For a quick look, set `SCALE = 1` (1080p) and `TEST_SECONDS = 8`.
- **Output location** → change `DRIVE_DIR` in cell 1; the notebook prints the exact Drive path in cell 6.